<h2>Import Libraries</h2>

In [ ]:
import pandas as pd
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import OneHotEncoder, LabelEncoder


<h2>Load Dataset</h2>

In [ ]:
df = pd.read_csv('../data/raw_data/loan_data_raw.csv')

<h2>Data Cleaning</h2>

In [ ]:
# Rename key columns, then standardize all column names to lowercase

rename_columns={
    'LoanAmount': 'loan_amount',
    'CoapplicantIncome': 'coapplicant_income',
    'ApplicantIncome': 'applicant_income'
}

df.rename(columns=rename_columns,inplace=True)

def standardize_columns(df):
    df.columns = (
        df.columns.str.strip()              
                 .str.lower()               
                 
    )
    return df

df = standardize_columns(df)
print(df.columns)

df.info()

In [ ]:
# Convert 3+ to 3 and cast dependents to a clean nullable integer

(df["dependents"] == "3+").sum()

df["dependents"] = df["dependents"].replace("3+", "3")

df["dependents"] = pd.to_numeric(df["dependents"])

df["dependents"] = df["dependents"].astype("Int64")

df["dependents"].unique()

df["dependents"].dtype

In [ ]:
# Convert credit_history to categorical dtype
df['credit_history'] = df['credit_history'].astype('category')

In [ ]:
# Impute missing values in categorical columns using the most frequent value

cat_cols = ['gender', 'married', 'dependents', 'self_employed', 'credit_history']
for col in cat_cols:
    cat_imputer = SimpleImputer(strategy='most_frequent')
    df[[col]] = cat_imputer.fit_transform(df[[col]])

In [ ]:
# Impute missing values in numeric columns using the median

num_cols = ['loan_amount', 'loan_amount_term']
num_imputer = SimpleImputer(strategy='median')
df[num_cols] = num_imputer.fit_transform(df[num_cols])

In [ ]:
print("Total missing values remaining:", df.isnull().sum())

<h2>Data Preprocessing</h2>

In [ ]:
# Combine base categorical columns with education and property_area
cat_features_all = cat_cols + ['education', 'property_area']

# Combine base numeric columns with applicant and coapplicant income 
num_cols_all = num_cols + ['applicant_income', 'coapplicant_income']

In [ ]:
# Fit and transform categorical columns into a one-hot encoded array

encoder = OneHotEncoder(
    sparse_output=False,
    drop='first',
    handle_unknown='ignore'
)
encoded_dat=encoder.fit_transform(df[cat_cols])
encoded_dat

In [ ]:
# Get encoded column names and label-encode the target variable

encoded_cols = encoder.get_feature_names_out(cat_cols)
target_encoder = LabelEncoder()
encoded_dat_loan_stat = target_encoder.fit_transform(
    df["loan_status"]
)

In [ ]:
df['loan_status'] = encoded_dat_loan_stat

In [ ]:
# Convert encoded array into a labeled integer DataFrame
cat_encoded_features  = pd.DataFrame(encoded_dat, columns=encoded_cols).astype(int)

In [ ]:
# Standardize column names to lowercase with underscores
cat_encoded_features.columns = cat_encoded_features.columns.str.lower().str.replace(' ', '_')

In [ ]:
# Reset index on encoded categorical features so rows align correctly when merging

cat_encoded_features = cat_encoded_features.reset_index(drop=True)
num_cleaned_features = df[num_cols_all].reset_index(drop=True)
target_feature = df['loan_status'].reset_index(drop=True)

In [ ]:
# Combine loan_id, encoded categorical features, numerical features, and target feature

df_final = pd.concat([
    df['loan_id'].reset_index(drop=True),
    cat_encoded_features,
    num_cleaned_features,
    target_feature
], axis=1)

df_final

<h2>Data Preprocessing</h2>